# Academic Survival Longitudinal Data Analysis

Exploratory analysis of `academic_survival_longitudinal (1).csv` using pandas and matplotlib.

Dataset: longitudinal student records with 22 columns (Student_ID through Censored), multiple semesters per student.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)


In [ ]:
# Load the CSV file
df = pd.read_csv('academic_survival_longitudinal (1).csv')

# Preview the first rows
df.head()


In [ ]:
# Dataset shape and column info
print('Shape:', df.shape)
print()
df.info()

# Summary statistics
df.describe(include='all')


In [ ]:
# Check for missing values per column
df.isnull().sum()


In [ ]:
# Map Gender to four standard categories: Female, Male, Others, Prefer not to say
df['Gender'] = df['Gender'].astype(str).str.strip().str.title()
gender_map = {
    'Female': 'Female', 'Male': 'Male', 'M': 'Male', 'F': 'Female',
    'Other': 'Others', 'Others': 'Others',
    'Prefer Not To Say': 'Prefer not to say',
}
df['Gender'] = df['Gender'].replace(gender_map)
df['Gender'] = df['Gender'].where(
    df['Gender'].isin(['Female', 'Male', 'Others', 'Prefer not to say']), 'Others'
)
df['Gender'].value_counts()


In [ ]:
# Handle negative Sem_GPA: flag invalid values (< 0), replace with NaN
df['sem_gpa_invalid'] = df['Sem_GPA'] < 0
print('Invalid (negative) Sem_GPA rows:')
print(df.loc[df['Sem_GPA'] < 0, ['Student_ID', 'Semester', 'Sem_GPA']])

df.loc[df['Sem_GPA'] < 0, 'Sem_GPA'] = np.nan
df['Sem_GPA'] = df['Sem_GPA'].clip(lower=0, upper=4)

df['Sem_GPA'].describe()


In [ ]:
# Impute missing Family_Income with median per student (longitudinal), fallback to overall median
df['Family_Income'] = df.groupby('Student_ID')['Family_Income'].transform(
    lambda s: s.fillna(s.median())
)
df['Family_Income'] = df['Family_Income'].fillna(df['Family_Income'].median())

print('Remaining missing Family_Income:', df['Family_Income'].isnull().sum())
df['Family_Income'].describe()


In [ ]:
# Handle missing Attendance (STU_00022 semester 2): impute with student median, fallback overall median
df['Attendance'] = df.groupby('Student_ID')['Attendance'].transform(
    lambda s: s.fillna(s.median())
)
df['Attendance'] = df['Attendance'].fillna(df['Attendance'].median())

print('Remaining missing Attendance:', df['Attendance'].isnull().sum())
df['Attendance'].describe()


In [ ]:
# Emergency_Expense outliers > 1,000: cap at the 99th percentile
outliers = df.loc[df['Emergency_Expense'] > 1000, ['Student_ID', 'Semester', 'Emergency_Expense']]
print('Emergency_Expense outliers > 1,000:')
print(outliers)

cap = df['Emergency_Expense'].quantile(0.99)
df['Emergency_Expense'] = df['Emergency_Expense'].clip(upper=cap)

print('Cap value (99th percentile):', cap)
df['Emergency_Expense'].describe()


In [ ]:
# Drop exact duplicate rows
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'Dropped {before - after} duplicate rows')

df.head()


In [ ]:
# End_of_Semester_Status categories: Enrolled, Dropped_Out, Graduated
print('End_of_Semester_Status value counts:')
print(df['End_of_Semester_Status'].value_counts())
print()
print('Censored value counts (binary 0/1):')
print(df['Censored'].value_counts())
print()
print('Target_Dropout_Next_Sem value counts:')
print(df['Target_Dropout_Next_Sem'].value_counts())


In [ ]:
# Semester distribution per student
print('Semesters per student:')
print(df.groupby('Student_ID')['Semester'].nunique().describe())
print()
print('Sample student trajectories:')
print(df[df['Student_ID'].isin(['STU_00001', 'STU_00018'])][['Student_ID', 'Semester', 'Sem_GPA', 'End_of_Semester_Status', 'Censored', 'Target_Dropout_Next_Sem']])


In [ ]:
# Distribution of Sem_GPA
df['Sem_GPA'].hist(bins=30)
plt.title('Distribution of Sem_GPA')
plt.xlabel('Sem_GPA')
plt.ylabel('Count')
plt.show()

# Scatter plot: Emergency_Expense vs Sem_GPA
df.plot.scatter(x='Emergency_Expense', y='Sem_GPA', alpha=0.5)
plt.title('Emergency_Expense vs Sem_GPA')
plt.show()

# End_of_Semester_Status by Gender
pd.crosstab(df['Gender'], df['End_of_Semester_Status'])


In [ ]:
# EDA Table 1: Gender Ã— End_of_Semester_Status crosstab with row percentages
t1 = pd.crosstab(df['Gender'], df['End_of_Semester_Status'], margins=True)
t1_pct = t1.div(t1['All'], axis=0).mul(100).round(1)
print('Table 1: Gender Ã— End_of_Semester_Status (% of row)')
print(t1_pct)
print()

# EDA Table 2: Semester Ã— End_of_Semester_Status crosstab
t2 = pd.crosstab(df['Semester'], df['End_of_Semester_Status'], margins=True)
print('Table 2: Semester Ã— End_of_Semester_Status (counts)')
print(t2)
print()

# EDA Table 3: Scholarship Ã— End_of_Semester_Status crosstab with row percentages
t3 = pd.crosstab(df['Scholarship'], df['End_of_Semester_Status'], margins=True)
t3_pct = t3.div(t3['All'], axis=0).mul(100).round(1)
print('Table 3: Scholarship Ã— End_of_Semester_Status (% of row)')
print(t3_pct)
print()

# EDA Table 4: Family_Income binned stats by End_of_Semester_Status
df['Income_Bin'] = pd.qcut(df['Family_Income'], q=4, duplicates='drop')
t4 = df.groupby(['Income_Bin', 'End_of_Semester_Status']).size().unstack(fill_value=0)
t4['Total'] = t4.sum(axis=1)
t4_pct = t4.div(t4['Total'], axis=0).mul(100).round(1)
print('Table 4: Income quartile Ã— End_of_Semester_Status (% of row)')
print(t4_pct)
print()

# EDA Table 5: Summary metrics by Gender
t5 = df.groupby('Gender').agg(
    n_students=('Student_ID', 'nunique'),
    n_rows=('Student_ID', 'size'),
    avg_sem_gpa=('Sem_GPA', 'mean'),
    avg_attendance=('Attendance', 'mean'),
    avg_financial_stress=('Financial_Stress', 'mean'),
    dropout_rate=('Target_Dropout_Next_Sem', 'mean'),
).round(3)
t5['dropout_rate'] = (t5['dropout_rate'] * 100).round(1)
print('Table 5: Summary metrics by Gender')
print(t5)


In [ ]:
# Distribution Tables (counts & percentages)
dist_status = df['End_of_Semester_Status'].value_counts()
dist_status_pct = (dist_status / len(df) * 100).round(1)
dist_status_df = pd.DataFrame({'Count': dist_status, 'Percent': dist_status_pct})
print('Distribution Table A: End_of_Semester_Status')
print(dist_status_df)
print()

dist_gender = df['Gender'].value_counts()
dist_gender_pct = (dist_gender / len(df) * 100).round(1)
dist_gender_df = pd.DataFrame({'Count': dist_gender, 'Percent': dist_gender_pct})
print('Distribution Table B: Gender')
print(dist_gender_df)
print()

dist_scholarship = df['Scholarship'].value_counts()
dist_scholarship_pct = (dist_scholarship / len(df) * 100).round(1)
dist_scholarship_df = pd.DataFrame({'Count': dist_scholarship, 'Percent': dist_scholarship_pct})
print('Distribution Table C: Scholarship')
print(dist_scholarship_df)
print()

dist_semester = df['Semester'].value_counts().sort_index()
dist_semester_pct = (dist_semester / len(df) * 100).round(1)
dist_semester_df = pd.DataFrame({'Count': dist_semester, 'Percent': dist_semester_pct})
print('Distribution Table D: Semester')
print(dist_semester_df)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution Tables', fontsize=16, weight='bold', y=1.02)

colors_status = {'Enrolled': '#059669', 'Dropped_Out': '#dc2626', 'Graduated': '#0891b2'}
ax = axes[0, 0]
bars = ax.bar(dist_status.index, dist_status.values, color=[colors_status.get(x, '#64748b') for x in dist_status.index])
ax.set_title('End_of_Semester_Status Distribution', weight='bold')
ax.set_ylabel('Count')
for i, (idx, val) in enumerate(dist_status.items()):
    ax.text(i, val/2, f'{dist_status_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=10)
ax.set_ylim(0, max(dist_status.values) * 1.15)

colors_g = {'Female': '#2563eb', 'Male': '#7c3aed', 'Others': '#d97706', 'Prefer not to say': '#be185d'}
ax = axes[0, 1]
bars = ax.bar(dist_gender.index, dist_gender.values, color=[colors_g.get(x, '#64748b') for x in dist_gender.index])
ax.set_title('Gender Distribution', weight='bold')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=15)
for i, (idx, val) in enumerate(dist_gender.items()):
    ax.text(i, val/2, f'{dist_gender_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=10)
ax.set_ylim(0, max(dist_gender.values) * 1.15)

ax = axes[1, 0]
labels_scholarship = ['No Scholarship (0)', 'Scholarship (1)']
bars = ax.bar(labels_scholarship, dist_scholarship.values, color=['#94a3b8', '#2563eb'])
ax.set_title('Scholarship Distribution', weight='bold')
ax.set_ylabel('Count')
for i, (idx, val) in enumerate(dist_scholarship.items()):
    ax.text(i, val/2, f'{dist_scholarship_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=10)
ax.set_ylim(0, max(dist_scholarship.values) * 1.15)

ax = axes[1, 1]
ax.bar(dist_semester.index, dist_semester.values, color='#0891b2')
ax.set_title('Semester Distribution', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Count')
ax.set_xticks(dist_semester.index)
for i, (idx, val) in enumerate(dist_semester.items()):
    ax.text(idx, val/2, f'{dist_semester_pct[idx]}%', ha='center', va='center', color='white', weight='bold', fontsize=9)
ax.set_ylim(0, max(dist_semester.values) * 1.15)

plt.tight_layout()
plt.savefig('eda_distribution_tables.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_distribution_tables.png')


In [ ]:
# Line Charts: trends over semesters
semester_stats = df.groupby('Semester').agg(
    dropout_rate=('Target_Dropout_Next_Sem', 'mean'),
    avg_gpa=('Sem_GPA', 'mean'),
    n=('Student_ID', 'size')
).reset_index()
semester_stats['dropout_rate_pct'] = semester_stats['dropout_rate'] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(semester_stats['Semester'], semester_stats['dropout_rate_pct'], marker='o', linewidth=2.5, markersize=8, color='#dc2626')
ax.set_title('Dropout Rate by Semester', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Dropout Rate (%)')
ax.set_xticks(semester_stats['Semester'])
ax.grid(True, alpha=0.3)
for _, row in semester_stats.iterrows():
    ax.annotate(f'{row["dropout_rate_pct"]:.1f}%', (row['Semester'], row['dropout_rate_pct']),
                textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9, weight='bold', color='#dc2626')

ax = axes[1]
ax.plot(semester_stats['Semester'], semester_stats['avg_gpa'], marker='s', linewidth=2.5, markersize=8, color='#059669')
ax.set_title('Average Sem_GPA by Semester', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Average Sem_GPA')
ax.set_xticks(semester_stats['Semester'])
ax.set_ylim(0, 4.2)
ax.grid(True, alpha=0.3)
for _, row in semester_stats.iterrows():
    ax.annotate(f'{row["avg_gpa"]:.2f}', (row['Semester'], row['avg_gpa']),
                textcoords="offset points", xytext=(0, 10), ha='center', fontsize=9, weight='bold', color='#059669')

plt.tight_layout()
plt.savefig('eda_line_charts_trends.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_line_charts_trends.png')

fig, ax = plt.subplots(figsize=(10, 6))
ax2 = ax.twinx()
line1 = ax.plot(semester_stats['Semester'], semester_stats['n'], marker='o', linewidth=2.5, markersize=8, color='#2563eb', label='Records')
line2 = ax2.plot(semester_stats['Semester'], semester_stats['dropout_rate_pct'], marker='s', linewidth=2.5, markersize=8, color='#dc2626', label='Dropout Rate (%)')
ax.set_title('Student Records and Dropout Rate by Semester', weight='bold')
ax.set_xlabel('Semester')
ax.set_ylabel('Number of Records', color='#2563eb')
ax2.set_ylabel('Dropout Rate (%)', color='#dc2626')
ax.set_xticks(semester_stats['Semester'])
ax.grid(True, alpha=0.3)
lines = line1 + line2
labels = [l.get_label() for l in lines]
ax.legend(lines, labels, loc='upper right')
plt.tight_layout()
plt.savefig('eda_line_chart_enrollment_dropout.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_line_chart_enrollment_dropout.png')


In [ ]:
# Bar Graphs: dropout rate by category
gender_dropout = df.groupby('Gender')['Target_Dropout_Next_Sem'].mean() * 100
gender_dropout = gender_dropout.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar1 = ['#dc2626' if v == gender_dropout.max() else '#2563eb' for v in gender_dropout.values]
bars = ax.bar(gender_dropout.index, gender_dropout.values, color=colors_bar1)
ax.set_title('Dropout Rate by Gender', weight='bold')
ax.set_ylabel('Dropout Rate (%)')
ax.set_ylim(0, max(gender_dropout.values) * 1.15)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}%', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_dropout_by_gender.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_dropout_by_gender.png')

scholarship_dropout = df.groupby('Scholarship')['Target_Dropout_Next_Sem'].mean() * 100
scholarship_dropout.index = ['No Scholarship (0)', 'Scholarship (1)']

fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(scholarship_dropout.index, scholarship_dropout.values, color=['#94a3b8', '#2563eb'])
ax.set_title('Dropout Rate by Scholarship', weight='bold')
ax.set_ylabel('Dropout Rate (%)')
ax.set_ylim(0, max(scholarship_dropout.values) * 1.15)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}%', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_dropout_by_scholarship.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_dropout_by_scholarship.png')

df['Income_Bin'] = pd.qcut(df['Family_Income'], q=4, duplicates='drop')
income_dropout = df.groupby('Income_Bin')['Target_Dropout_Next_Sem'].mean() * 100

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(range(len(income_dropout)), income_dropout.values, color=['#dc2626', '#ea580c', '#059669', '#0891b2'])
ax.set_title('Dropout Rate by Income Quartile', weight='bold')
ax.set_ylabel('Dropout Rate (%)')
ax.set_xlabel('Income Quartile (Q1=Lowest â†’ Q4=Highest)')
ax.set_xticks(range(len(income_dropout)))
ax.set_xticklabels(['Q1\n(0-35k)', 'Q2\n(35-52k)', 'Q3\n(52-77k)', 'Q4\n(77k+)'], fontsize=10)
ax.set_ylim(0, max(income_dropout.values) * 1.15)
for i, bar in enumerate(bars):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.1, f'{h:.1f}%', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_dropout_by_income.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_dropout_by_income.png')

gender_stress = df.groupby('Gender')['Financial_Stress'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.bar(gender_stress.index, gender_stress.values, color=['#2563eb', '#7c3aed', '#d97706', '#be185d'])
ax.set_title('Average Financial Stress by Gender', weight='bold')
ax.set_ylabel('Average Financial Stress (1-5 scale)')
ax.set_ylim(0, max(gender_stress.values) * 1.15)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.02, f'{h:.2f}', ha='center', va='bottom', fontsize=11, weight='bold')
plt.tight_layout()
plt.savefig('eda_bar_stress_by_gender.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved eda_bar_stress_by_gender.png')


In [ ]:
# ========== FEATURE ENGINEERING WITH UNIFIED CATEGORICAL ENCODING ==========
df_fe = df.copy()
df_fe = df_fe.sort_values(['Student_ID', 'Semester']).reset_index(drop=True)

# --- Unified Categorical Encoding (Ordinal) ---
# Define explicit orderings for each categorical feature
cat_encodings = {
    'Gender': {'Prefer not to say': 0, 'Others': 1, 'F': 2, 'female': 2, 'Female': 2, 'M': 3, 'male': 3, 'Male': 3},
    'Housing_Status': {'With Parents': 0, 'Off-Campus': 1, 'On-Campus': 2},
    'End_of_Semester_Status': {'Dropped_Out': 0, 'Enrolled': 1, 'Graduated': 2}
}

# Apply mappings - unmapped values become NaN (then fill with median)
for col, mapping in cat_encodings.items():
    if col in df_fe.columns:
        df_fe[f'{col}_encoded'] = df_fe[col].map(mapping)
        df_fe[f'{col}_encoded'] = df_fe[f'{col}_encoded'].fillna(df_fe[f'{col}_encoded'].median())
        print(f'{col}: mapped {df_fe[col].nunique()} categories -> {col}_encoded')

# --- Lag features (previous semester values) ---
lag_cols = ['Sem_GPA', 'Attendance', 'Financial_Stress', 'Emergency_Expense',
            'Work_Hours', 'Course_Load', 'LMS_Logins', 'Advising_Visits',
            'Failed_Courses']
for col in lag_cols:
    if col in df_fe.columns:
        df_fe[f'{col}_lag1'] = df_fe.groupby('Student_ID')[col].shift(1)
        df_fe[f'{col}_lag2'] = df_fe.groupby('Student_ID')[col].shift(2)

# --- Rolling statistics (window=2,3) ---
for col in ['Sem_GPA', 'Attendance', 'Financial_Stress']:
    if col in df_fe.columns:
        df_fe[f'{col}_roll2_mean'] = df_fe.groupby('Student_ID')[col].transform(lambda x: x.rolling(2, min_periods=1).mean())
        df_fe[f'{col}_roll2_std'] = df_fe.groupby('Student_ID')[col].transform(lambda x: x.rolling(2, min_periods=1).std())
        df_fe[f'{col}_roll3_mean'] = df_fe.groupby('Student_ID')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())
        df_fe[f'{col}_roll3_std'] = df_fe.groupby('Student_ID')[col].transform(lambda x: x.rolling(3, min_periods=1).std())

# --- Trend features (difference from previous semester) ---
for col in ['Sem_GPA', 'Attendance', 'Financial_Stress']:
    if col in df_fe.columns:
        df_fe[f'{col}_diff1'] = df_fe.groupby('Student_ID')[col].diff(1)
        df_fe[f'{col}_diff2'] = df_fe.groupby('Student_ID')[col].diff(2)

# --- Cumulative features ---
df_fe['cum_failed_courses'] = df_fe.groupby('Student_ID')['Failed_Courses'].cumsum()
df_fe['cum_emergency_expense'] = df_fe.groupby('Student_ID')['Emergency_Expense'].cumsum()
df_fe['cum_advising_visits'] = df_fe.groupby('Student_ID')['Advising_Visits'].cumsum()

# --- Interaction features ---
df_fe['work_study_ratio'] = df_fe['Work_Hours'] / (df_fe['Course_Load'] + 1)
df_fe['stress_income_ratio'] = df_fe['Financial_Stress'] / (df_fe['Family_Income'] / 10000 + 1)
df_fe['gpa_attendance_interaction'] = df_fe['Sem_GPA'] * df_fe['Attendance']
df_fe['scholarship_gpa_interaction'] = df_fe['Scholarship'] * df_fe['Sem_GPA']

# --- Semester position features ---
df_fe['semester_since_start'] = df_fe.groupby('Student_ID').cumcount() + 1
df_fe['total_semesters_per_student'] = df_fe.groupby('Student_ID')['Semester'].transform('nunique')
df_fe['semester_progress_pct'] = df_fe['semester_since_start'] / df_fe['total_semesters_per_student']

# Target: next semester dropout
df_fe['target'] = df_fe['Target_Dropout_Next_Sem']

print(f'\nOriginal features: {len(df.columns)}')
print(f'Engineered features: {len(df_fe.columns)}')
print(f'New features added: {len(df_fe.columns) - len(df.columns)}')

# Save for feature selection
import pickle
with open('df_fe.pkl', 'wb') as f:
    pickle.dump(df_fe, f)
print('Saved df_fe.pkl for feature selection')


In [ ]:
# ========== FEATURE SELECTION ==========
import pickle
with open('df_fe.pkl', 'rb') as f:
    df_fe = pickle.load(f)

# Prepare features and target
exclude_cols = ['Student_ID', 'End_of_Semester_Status', 'Target_Dropout_Next_Sem',
                'Censored', 'Income_Bin', 'sem_gpa_invalid', 'target',
                'Gender', 'Housing_Status']  # use encoded versions instead
feature_cols = [c for c in df_fe.columns if c not in exclude_cols and df_fe[c].dtype in ['int64', 'float64', 'int32', 'float32']]

# Drop rows with NaN in features (from lag/rolling)
df_fs = df_fe[feature_cols + ['target']].dropna()
X = df_fs[feature_cols]
y = df_fs['target']

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')

# 1. Correlation with target
corr_with_target = X.corrwith(y).abs().sort_values(ascending=False)
print('\nTop 20 features by correlation with target:')
print(corr_with_target.head(20))

# 2. Feature correlation matrix (multicollinearity check)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = [(col, row) for col in upper.columns for row in upper.index if upper.loc[row, col] > 0.9]
print(f'\nHighly correlated feature pairs (>0.9): {len(high_corr_pairs)}')
for pair in high_corr_pairs[:10]:
    print(f'  {pair[0]} - {pair[1]}: {corr_matrix.loc[pair[0], pair[1]]:.3f}')

# 3. Mutual Information
from sklearn.feature_selection import mutual_info_classif
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({'feature': X.columns, 'mi_score': mi_scores})
mi_df = mi_df.sort_values('mi_score', ascending=False)
print('\nTop 20 features by Mutual Information:')
print(mi_df.head(20))

# 4. Random Forest Feature Importance
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
rf.fit(X, y)
importance_df = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_})
importance_df = importance_df.sort_values('importance', ascending=False)
print('\nTop 20 features by Random Forest Importance:')
print(importance_df.head(20))

# 5. Select top features combining all methods (voting >= 2)
top_corr = set(corr_with_target.head(30).index)
top_mi = set(mi_df.head(30)['feature'].values)
top_rf = set(importance_df.head(30)['feature'].values)

all_top = list(top_corr | top_mi | top_rf)
feature_votes = {}
for f in all_top:
    votes = sum([f in top_corr, f in top_mi, f in top_rf])
    feature_votes[f] = votes

selected_features = [f for f, v in feature_votes.items() if v >= 2]
print(f'\nFeatures selected by >=2 methods: {len(selected_features)}')
for f in sorted(selected_features, key=lambda x: feature_votes[x], reverse=True):
    print(f'  {f}: {feature_votes[f]} votes')

# Save selected features
with open('selected_features.pkl', 'wb') as f:
    pickle.dump(selected_features, f)
with open('feature_importance.pkl', 'wb') as f:
    pickle.dump({'correlation': corr_with_target, 'mi': mi_df.set_index('feature')['mi_score'], 'rf': importance_df.set_index('feature')['importance']}, f)
print('\nSaved selected_features.pkl and feature_importance.pkl')

# 6. Visualize feature importance
fig, axes = plt.subplots(1, 3, figsize=(18, 8))

top15_corr = corr_with_target.head(15)
axes[0].barh(range(len(top15_corr)), top15_corr.values, color='#2563eb')
axes[0].set_yticks(range(len(top15_corr)))
axes[0].set_yticklabels(top15_corr.index, fontsize=9)
axes[0].set_title('Top 15: Correlation with Target', weight='bold')
axes[0].invert_yaxis()

top15_mi = mi_df.head(15)
axes[1].barh(range(len(top15_mi)), top15_mi['mi_score'].values, color='#059669')
axes[1].set_yticks(range(len(top15_mi)))
axes[1].set_yticklabels(top15_mi['feature'].values, fontsize=9)
axes[1].set_title('Top 15: Mutual Information', weight='bold')
axes[1].invert_yaxis()

top15_rf = importance_df.head(15)
axes[2].barh(range(len(top15_rf)), top15_rf['importance'].values, color='#dc2626')
axes[2].set_yticks(range(len(top15_rf)))
axes[2].set_yticklabels(top15_rf['feature'].values, fontsize=9)
axes[2].set_title('Top 15: Random Forest Importance', weight='bold')
axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig('feature_selection_importance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print('Saved feature_selection_importance.png')


In [ ]:
# ========== LOGISTIC REGRESSION MODEL (NO LEAKAGE) ==========
import pickle
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (roc_auc_score, average_precision_score, 
                             f1_score, precision_score, recall_score,
                             brier_score_loss, classification_report,
                             confusion_matrix)

# Load engineered features
with open('df_fe.pkl', 'rb') as f:
    df_fe = pickle.load(f)

# Load selected features
with open('selected_features.pkl', 'rb') as f:
    selected_features = pickle.load(f)

# Prepare data with ONE-HOT encoding for nominal categoricals
# Nominal: Gender, Housing_Status -> one-hot
# EXCLUDE: End_of_Semester_Status_encoded (leakage - current status predicting next semester)
df_model = df_fe.copy()

# One-hot encode nominal categoricals
df_model = pd.get_dummies(df_model, 
                          columns=['Gender', 'Housing_Status'], 
                          prefix=['Gender', 'Housing'], 
                          drop_first=False,
                          dtype=int)

# Identify feature columns (selected + one-hot expanded)
onehot_cols = [c for c in df_model.columns if c.startswith('Gender_') or c.startswith('Housing_')]
feat_cols = [c for c in selected_features if c in df_model.columns] + onehot_cols

# Exclude raw categoricals, leakage columns, and End_of_Semester_Status_encoded
exclude = ['Student_ID', 'End_of_Semester_Status', 'Target_Dropout_Next_Sem',
           'Censored', 'Income_Bin', 'sem_gpa_invalid', 'target',
           'Gender', 'Housing_Status', 'End_of_Semester_Status_encoded']
feat_cols = [c for c in feat_cols if c not in exclude and df_model[c].dtype in ['int64', 'float64', 'int32', 'float32', 'uint8']]

# Drop NaN rows (from lag/rolling features)
df_clean = df_model[feat_cols + ['target', 'Student_ID']].dropna()
X = df_clean[feat_cols]
y = df_clean['target']
groups = df_clean['Student_ID']

print(f'Feature matrix: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')
print(f'Unique students: {groups.nunique()}')
print(f'Features: {len(feat_cols)}')

# Pipeline: StandardScaler + LogisticRegression
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42,
        solver='lbfgs',
        C=1.0
    ))
])

# Grouped Cross-Validation (5 folds)
gkf = GroupKFold(n_splits=5)

# Cross-validated predictions for evaluation
y_pred_proba = cross_val_predict(pipe, X, y, cv=gkf, groups=groups, 
                                 method='predict_proba', n_jobs=-1)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

# Metrics
roc_auc = roc_auc_score(y, y_pred_proba)
pr_auc = average_precision_score(y, y_pred_proba)
f1 = f1_score(y, y_pred)
precision = precision_score(y, y_pred)
recall = recall_score(y, y_pred)
brier = brier_score_loss(y, y_pred_proba)

print('\n========== MODEL PERFORMANCE (GroupKFold CV) - NO LEAKAGE ==========')
print(f'ROC-AUC:       {roc_auc:.4f}')
print(f'PR-AUC:        {pr_auc:.4f}')
print(f'F1 Score:      {f1:.4f}')
print(f'Precision:     {precision:.4f}')
print(f'Recall:        {recall:.4f}')
print(f'Brier Score:   {brier:.4f}')
print(f'Baseline PR-AUC (random): {y.mean():.4f}')

print('\nClassification Report:')
print(classification_report(y, y_pred, target_names=['Retained', 'Dropout']))

print('Confusion Matrix:')
cm = confusion_matrix(y, y_pred)
print(cm)

# Fit on full data for coefficient inspection
pipe.fit(X, y)

# Feature coefficients
coef_df = pd.DataFrame({
    'feature': feat_cols,
    'coefficient': pipe.named_steps['model'].coef_[0]
})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False)

print('\n========== TOP 20 FEATURE COEFFICIENTS ==========')
print(coef_df.head(20).to_string(index=False))

# Save model
with open('lr_model_no_leakage.pkl', 'wb') as f:
    pickle.dump(pipe, f)
print('\nSaved lr_model_no_leakage.pkl')

# ========== VISUALIZATIONS ==========
import matplotlib.pyplot as plt

# 1. Coefficient plot
top15_coef = coef_df.head(15)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#dc2626' if c < 0 else '#059669' for c in top15_coef['coefficient']]
ax.barh(range(len(top15_coef)), top15_coef['coefficient'], color=colors)
ax.set_yticks(range(len(top15_coef)))
ax.set_yticklabels(top15_coef['feature'], fontsize=10)
ax.set_xlabel('Coefficient (log-odds)')
ax.set_title('Top 15 Logistic Regression Coefficients (No Leakage)', weight='bold')
ax.axvline(0, color='black', linewidth=0.5)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('lr_coefficients_no_leakage.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 2. ROC Curve
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
RocCurveDisplay.from_predictions(y, y_pred_proba, ax=axes[0], name='Logistic Regression')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
axes[0].set_title(f'ROC Curve (AUC = {roc_auc:.3f})', weight='bold')

PrecisionRecallDisplay.from_predictions(y, y_pred_proba, ax=axes[1], name='Logistic Regression')
axes[1].axhline(y.mean(), color='k', linestyle='--', alpha=0.5, label=f'Baseline ({y.mean():.3f})')
axes[1].legend()
axes[1].set_title(f'PR Curve (AUC = {pr_auc:.3f})', weight='bold')

plt.tight_layout()
plt.savefig('lr_roc_pr_curves_no_leakage.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

# 3. Calibration plot
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(figsize=(8, 6))
prob_true, prob_pred = calibration_curve(y, y_pred_proba, n_bins=10, strategy='quantile')
ax.plot(prob_pred, prob_true, 'o-', label='Logistic Regression')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfectly Calibrated')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curve', weight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('lr_calibration_no_leakage.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

print('Saved: lr_coefficients_no_leakage.png, lr_roc_pr_curves_no_leakage.png, lr_calibration_no_leakage.png')


In [ ]:
# Save cleaned data
df.to_csv('academic_survival_longitudinal_cleaned.csv', index=False)
print('Saved cleaned data to academic_survival_longitudinal_cleaned.csv')
print('Final shape:', df.shape)


## Next steps

- Confirm `academic_survival_longitudinal (1).csv` is in this folder before running.
- Install dependencies: `conda install pandas matplotlib`
- Review invalid Sem_GPA rows flagged in the cleaning cell.
- Adjust outlier capping / imputation strategy as needed.
- Add statistical tests or survival modeling using `Censored` and `Target_Dropout_Next_Sem`.
